# Dataset verification — collector 15s CSVs vs freqtrade (Binance futures) 5-min candles

Loads **all** collector CSVs (like run.005), sorts by timestamp (files are **not** in date order),
reports the data period and every gap, aggregates the 15s bars into **5-minute OHLCV**
(20 x 15s per bucket), downloads the matching Binance USDT-M futures 5m candles (the same
source freqtrade uses, via `ccxt`), and compares them.

**`PRICE` mode (Cell 2):** `mid` (bid/ask midpoint), `vwap` (qty-weighted trade VWAP), or
`hybrid` (trade VWAP for open/close, bid/ask max/min for high/low). Open/close match freqtrade's
trade candles to ~0.02 bps in all modes; for **high/low** `mid`/`hybrid` are truer because a 15s
VWAP averages away the sub-15s wick extremes (`vwap` compresses H/L by ~2 bps).

**Timezone (important):** in these files `future_timestamp` is a **UTC epoch**, but the
`future_datetime` **string is local time (UTC+3)**. Everything below aligns on the epoch as UTC.

In [ ]:
# ── Cell 1: deps ─────────────────────────────────────────────────────────
!pip install -q ccxt
import glob, os, numpy as np, pandas as pd, ccxt
print('ccxt', ccxt.__version__)

In [ ]:
# ── Cell 2: config + (optional) mount Drive & extract data ───────────────
DATA_DIR = '/root/btc_data'      # folder that will hold the *.csv / *.csv.gz files
SYMBOL   = 'BTC/USDT:USDT'       # Binance USDT-M perpetual (ccxt unified symbol)
SEP      = ';'
WIN      = 15                    # collector bar size (seconds)
BATCH    = 20                    # 15s rows per 5-min bucket (20*15 = 300s)
INTERVAL = 300                   # seconds per output bar
PRICE    = 'hybrid'              # 'mid' | 'vwap' | 'hybrid'  (how to price the 15s bars)
                                #   mid    = bid/ask midpoint for O/H/L/C
                                #   vwap   = qty-weighted trade VWAP for O/H/L/C
                                #   hybrid = trade VWAP for O/C, bid/ask max/min for H/L (truest)

# --- Colab: mount Drive and unpack the tar (comment out if files already local) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    TAR = '/content/drive/MyDrive/btc_data.tar.xz'   # <-- adjust path
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(TAR):
        !tar -xf "{TAR}" -C "{DATA_DIR}"
except Exception as e:
    print('skip Drive mount:', e)

files = sorted(glob.glob(os.path.join(DATA_DIR, '**', '*.csv*'), recursive=True))
print(f'found {len(files)} csv files')

In [ ]:
# ── Cell 3: load ALL files (only the columns we need) ────────────────────
NEED = ['future_timestamp','future_datetime',
        'future_bid_open','future_ask_open','future_bid_close','future_ask_close',
        'future_bid_min','future_ask_min','future_bid_max','future_ask_max',
        'future_buy_vwap','future_sell_vwap','future_buy_qty','future_sell_qty']

def load_one(p):
    hdr  = pd.read_csv(p, sep=SEP, nrows=0).columns
    cols = [c for c in NEED if c in hdr]
    if 'future_timestamp' not in cols:
        return None
    return pd.read_csv(p, sep=SEP, usecols=cols)   # pandas infers .gz from extension

frames = []
for p in files:
    try:
        d = load_one(p)
        if d is not None and len(d):
            frames.append(d)
    except Exception as e:
        print('  skip', os.path.basename(p), e)
raw = pd.concat(frames, ignore_index=True)
print(f'loaded {len(raw):,} rows from {len(frames)} files')

In [ ]:
# ── Cell 4: TIMEZONE + sort (files are NOT date-ordered) + dedupe ────────
raw['ts'] = raw['future_timestamp'].astype('int64')
raw['dt'] = pd.to_datetime(raw['ts'], unit='s', utc=True)          # epoch treated as UTC

if 'future_datetime' in raw.columns:
    off = (pd.to_datetime(raw['future_datetime'].iloc[0])
           - raw['dt'].dt.tz_localize(None).iloc[0]).total_seconds()/3600
    print(f'timezone check: future_datetime string is UTC{off:+.0f}h vs epoch  -> aligning on epoch=UTC')

df = raw.sort_values('ts').drop_duplicates('ts').reset_index(drop=True)
print(f'after sort+dedupe: {len(df):,} rows')

In [ ]:
# ── Cell 5: data period + gap report ─────────────────────────────────────
print(f'PERIOD (UTC): {df.dt.iloc[0]}  ->  {df.dt.iloc[-1]}')
span = int(df.ts.iloc[-1] - df.ts.iloc[0])
print(f'span: {span/3600:.1f} h = {span/86400:.2f} days')
print(f'expected 15s rows if continuous: {span//WIN + 1:,}   actual: {len(df):,}   '
      f'coverage: {len(df)/(span//WIN+1)*100:.1f}%')

gap = df.ts.diff()
holes = df[gap > WIN]
print(f'\nGAP PERIODS (>{WIN}s): {len(holes)}')
tot_missing = 0
for i in holes.index:
    g = int(gap.loc[i]); tot_missing += g - WIN
    print(f'   {g:>8,}s ({g/3600:6.2f} h)   {df.dt.loc[i-1]}  ->  {df.dt.loc[i]}')
print(f'\ntotal missing time: {tot_missing/3600:.1f} h across {len(holes)} gaps')

In [ ]:
# ── Cell 6: aggregate 15s rows -> 5-min OHLCV, by UTC bucket ─────────────
# Bucket = floor(ts/300)*300 = candle OPEN time (UTC), the same convention as
# Binance/freqtrade. A complete bucket holds BATCH(=20) rows; partial buckets
# (start/end of a session or across a gap) are flagged.
#
# Per-15s price series (see PRICE in Cell 2):
#   mid_* = bid/ask midpoint  (true 15s quote extremes in max/min)
#   vwap  = qty-weighted trade VWAP of both sides, falling back to mid when a
#           15s bar had no trades (VWAP averages within 15s, so it compresses
#           wicks -> use it for O/C, not H/L)
mid_o = (df.future_bid_open  + df.future_ask_open )/2
mid_c = (df.future_bid_close + df.future_ask_close)/2
mid_h = (df.future_bid_max   + df.future_ask_max  )/2
mid_l = (df.future_bid_min   + df.future_ask_min  )/2
bq, sq = df.future_buy_qty, df.future_sell_qty
tot  = bq + sq
vwap = (df.future_buy_vwap*bq + df.future_sell_vwap*sq) / tot.replace(0, np.nan)
n_notrade = int(vwap.isna().sum())
vwap = vwap.fillna(mid_c)

if PRICE == 'mid':
    o, h, l, c = mid_o, mid_h, mid_l, mid_c
elif PRICE == 'vwap':
    o, h, l, c = vwap,  vwap,  vwap,  vwap
elif PRICE == 'hybrid':
    o, h, l, c = vwap,  mid_h, mid_l, vwap      # trade O/C, quote H/L
else:
    raise ValueError(f"PRICE must be mid|vwap|hybrid, got {PRICE!r}")
print(f"PRICE mode: {PRICE}   (15s bars with no trades, VWAP->mid fallback: {n_notrade})")

b = pd.DataFrame({'ts':df.ts,'o':o,'c':c,'h':h,'l':l,'v':tot})
b['bucket'] = (b.ts // INTERVAL) * INTERVAL
g = b.groupby('bucket')
me = pd.DataFrame({'open':g.o.first(),'high':g.h.max(),'low':g.l.min(),
                   'close':g.c.last(),'volume':g.v.sum(),'n':g.size()})
me.index = pd.to_datetime(me.index, unit='s', utc=True)
print(f'5-min buckets: {len(me):,}   complete(n={BATCH}): {(me.n==BATCH).sum():,}   '
      f'partial: {(me.n<BATCH).sum():,}')
me.head()

In [ ]:
# ── Cell 7: download the SAME period from Binance futures (freqtrade source)
ex = ccxt.binance({'options': {'defaultType': 'future'}, 'enableRateLimit': True})
start_ms = int(me.index.min().timestamp()*1000)
end_ms   = int(me.index.max().timestamp()*1000) + INTERVAL*1000
rows, since = [], start_ms
while since < end_ms:
    o = ex.fetch_ohlcv(SYMBOL, '5m', since=since, limit=1000)
    if not o:
        break
    rows += o
    since = o[-1][0] + INTERVAL*1000
    if len(o) < 1000:
        break
ft = pd.DataFrame(rows, columns=['ts','open','high','low','close','volume'])
ft['date'] = pd.to_datetime(ft['ts'], unit='ms', utc=True)
ft = ft.drop_duplicates('ts').set_index('date').sort_index()
ft = ft[(ft.index >= me.index.min()) & (ft.index <= me.index.max())]
print(f'freqtrade/Binance 5m candles: {len(ft):,}   {ft.index.min()} -> {ft.index.max()}')

In [ ]:
# ── Cell 8: compare (mid-OHLCV vs trade-OHLCV) ───────────────────────────
j = me.join(ft[['open','high','low','close','volume']], lsuffix='_me', rsuffix='_ft', how='inner')
comp = j[j.n == BATCH]                       # only complete buckets for a fair check
print(f'overlapping 5-min bars: {len(j):,}   (complete used for stats: {len(comp):,})')
missing_ft = me.index.difference(ft.index)
print(f'buckets we have but Binance does not: {len(missing_ft)}')
print()
for col in ['open','high','low','close']:
    d = (comp[f'{col}_me'] - comp[f'{col}_ft']) / comp[f'{col}_ft'] * 1e4
    print(f'  {col:5s}  median {d.median():+6.2f} bps   mean {d.mean():+6.2f}   '
          f'p95|Δ| {d.abs().quantile(.95):6.2f}   max|Δ| {d.abs().max():6.2f} bps')
vr = comp.volume_me / comp.volume_ft
print(f'  volume ratio (collector/freqtrade)  median {vr.median():.3f}  mean {vr.mean():.3f}  '
      f'p05 {vr.quantile(.05):.3f}  p95 {vr.quantile(.95):.3f}')

BPS_OK, VOL_OK = 5.0, 0.05                    # tolerances for "same data"
close_ok = abs((comp.close_me-comp.close_ft)/comp.close_ft*1e4).median() < BPS_OK
vol_ok   = abs(vr.median()-1) < VOL_OK
print(f'\nVERDICT: close median <{BPS_OK}bps: {close_ok}   volume median ~1: {vol_ok}   '
      f'=> dataset {"MATCHES freqtrade ✓" if (close_ok and vol_ok) else "MISMATCH — investigate ✗"}')

In [ ]:
# ── Cell 9: plots — overlay + per-bar close difference ───────────────────
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(13, 7), height_ratios=[2,1], sharex=True)
ax[0].plot(comp.index, comp.close_ft, lw=0.8, label='freqtrade close (trade)')
ax[0].plot(comp.index, comp.close_me, lw=0.8, ls='--', label='collector close (mid)')
ax[0].legend(); ax[0].set_title('5-min close: collector vs freqtrade'); ax[0].grid(alpha=.3)
d = (comp.close_me-comp.close_ft)/comp.close_ft*1e4
ax[1].plot(comp.index, d, lw=0.6); ax[1].axhline(0, color='k', lw=.5)
ax[1].set_title('close difference (bps)'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()